In [1]:
!pip show openai langchain llama-index faiss-cpu

Name: openai
Version: 2.31.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: langchain-openai, llama-index-embeddings-openai, llama-index-llms-openai
---
Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: llama-index
Version: 0.14.21
Summary: Interface between LLMs and your data
Home-page: https://llamaindex.ai
Author: 
Author-email: Jerry Liu <jerry@llamaindex.ai>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: llama-index-core, llama-index-embeddings-openai

In [2]:
!pip install -U llama-index faiss-cpu langchain-openai

In [6]:
!pip install llama-index-vector-stores-faiss

In [73]:
!pip install llama-parse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 14.0 MB/s eta 0:00:00


In [1]:
!pip install llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 12.5 MB/s eta 0:00:00


In [21]:
from getpass import getpass
import os
os.environ["OPENAI_API_KEY"] = getpass("Enter api key: ")

Enter api key: ··········


In [3]:
from google.colab import files
document = files.upload()

Saving AITools_Unit-1.pdf to AITools_Unit-1.pdf


In [4]:
file_path = list(document.keys())[0]
from llama_index.readers.file import PDFReader

documents = PDFReader().load_data(file = file_path)

In [5]:
print(len(documents))
print(documents[0].text[:500])

14
UNIT-1 
Introduction to Artificial Intelligence: What is AI, Foundations of AI, Goals of AI, and Applications of 
AI. 
 
Q) Define AI. Describe the organization of AI Definition. 
John McCarthy in mid -1950’scoined the term ― Artificial Intelligence ‖ which he 
would define as ―the science and engineering of making intelligent machines‖ 
AI is about teaching the machines to learn, to act, and t hink as humans would 
do. We can organize AI definition into 4 categories: 
 
 The definitions on top


In [10]:
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(chunk_size=512,chunk_overlap=50)
index = VectorStoreIndex.from_documents(
    documents,
    transformers = [parser]

)


In [11]:
from llama_index.core.indices import vector_store
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
import faiss

dimension = 1536
faiss_index = faiss.IndexFlatL2(dimension)

vector_store = FaissVectorStore(faiss_index=faiss_index)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context

)



In [13]:
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
"""
You are a helpful learning assistant.
Rules:
1. Answer ONLY using the provided document.
2. if relevant partial information is present try to explain it as much as possible.
3. Keep answer short and simple (like you are explaining to a 5 year old.)
4. ONLY say "i don't know" when nothing relevant is found.

Cntext:
{context_str}

Question:
{query_str}

Answer:
"""
)

In [14]:
from llama_index.core.response_synthesizers import get_response_synthesizer
response_synthesizer = get_response_synthesizer(
    text_qa_template= qa_prompt
)

In [20]:
query_engine = index.as_query_engine(
    similarity_top_k=2,
    response_synthesizer=response_synthesizer
    )

In [19]:
while True:
  q=input("Ask: ")

  if q.lower()=="exit":
    print("Chat ended")
    break
  print("Bot: ", query_engine.query(q))

Ask: what is ai
Bot:  AI stands for Artificial Intelligence, which is the science and engineering of making intelligent machines that can learn, act, and think like humans.
Ask: what is hindi
Bot:  I don't know.
Ask: exit
Chat ended
